# 电催化剂设计案例研究
# Electrocatalyst Design Case Study

本Notebook演示如何设计ORR/OER双功能电催化剂

**流程概览：**
1. 构建金属表面模型 (111, 100, 110晶面)
2. DFT计算吸附能 (O*, OH*, OOH*)
3. Scaling relation分析
4. ORR/OER过电位计算
5. 火山图绘制与活性预测
6. 双功能催化剂推荐

## 1. 环境设置与导入

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime

# 导入自定义模块
sys.path.insert(0, '/root/.openclaw/workspace/dft_lammps_research')
sys.path.insert(0, '/root/.openclaw/workspace/dft_lammps_research/applications/catalyst')

from case_catalyst import CatalystDesignConfig, ElectrocatalystDesigner

# 科学计算库
from ase import Atoms
from ase.build import fcc111, fcc100, bcc110, hcp0001, bulk
from ase.io import write

# 可视化
import matplotlib.pyplot as plt
import seaborn as sns

# 设置可视化样式
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

print("✓ 环境设置完成")
print(f"时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 2. 配置参数设置

In [ ]:
# 创建配置
config = CatalystDesignConfig(
    reaction="both",  # ORR + OER
    metals=["Pt", "Pd", "Au", "Ag", "Cu", "Ni", "Co", "Ir", "Ru", "Rh"],
    surface_facets=["111", "100", "110"],
    slab_thickness=6.0,
    vacuum_thickness=15.0,
    work_dir="./catalyst_design_results",
)

print("催化剂设计配置:")
print(f"  目标反应: {config.reaction}")
print(f"  分析金属: {config.metals}")
print(f"  表面晶面: {config.surface_facets}")
print(f"  吸附物种: {config.adsorbates}")
print(f"  工作目录: {config.work_dir}")

## 3. Phase 1: 构建金属表面模型

In [ ]:
# 创建催化剂设计器
designer = ElectrocatalystDesigner(config)

# 构建表面模型
surfaces = designer.build_surface_models()

print(f"\n共构建 {len(surfaces)} 个表面模型\n")

# 显示表面信息
surface_info = []
for name, atoms in surfaces.items():
    metal, facet = name.rsplit('_', 1)
    surface_info.append({
        'Surface': name,
        'Metal': metal,
        'Facet': facet,
        'N_atoms': len(atoms),
        'Cell_a': atoms.cell[0, 0],
        'Cell_b': atoms.cell[1, 1]
    })

df_surfaces = pd.DataFrame(surface_info)
display(df_surfaces.head(15))

## 4. 可视化表面结构

In [ ]:
# 绘制表面原子分布
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, (name, atoms) in enumerate(list(surfaces.items())[:6]):
    ax = axes[idx]
    
    # 获取原子位置
    pos = atoms.positions
    
    # 按z坐标着色
    z_coords = pos[:, 2]
    
    scatter = ax.scatter(pos[:, 0], pos[:, 1], c=z_coords, 
                        s=100, cmap='viridis', edgecolors='black')
    ax.set_title(f'{name}', fontsize=14, fontweight='bold')
    ax.set_xlabel('x (Å)')
    ax.set_ylabel('y (Å)')
    ax.set_aspect('equal')
    plt.colorbar(scatter, ax=ax, label='z (Å)')

plt.suptitle('Metal Surface Structures (Top View)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("✓ 表面结构可视化完成")

## 5. Phase 2: 计算吸附能

In [ ]:
# 计算吸附能 (使用模拟数据)
adsorption_energies = designer.calculate_adsorption_energies(skip_dft=True)

# 创建吸附能数据表
ads_data = []
for surface, energies in adsorption_energies.items():
    ads_data.append({
        'Surface': surface,
        'dG_O*': energies.get('O', 0),
        'dG_OH*': energies.get('OH', 0),
        'dG_OOH*': energies.get('OOH', 0),
        'dG_H*': energies.get('H', 0)
    })

df_ads = pd.DataFrame(ads_data)
df_ads = df_ads.sort_values('dG_OH*')

print("\n吸附能数据 (eV):\n")
display(df_ads.head(15))

## 6. Scaling Relation 分析

In [ ]:
# 绘制Scaling relation
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# O* vs OH*
ax = axes[0]
x = df_ads['dG_OH*']
y = df_ads['dG_O*']

# 线性拟合
coeffs = np.polyfit(x, y, 1)
fit_line = np.poly1d(coeffs)
x_fit = np.linspace(x.min(), x.max(), 100)

ax.plot(x_fit, fit_line(x_fit), 'r--', linewidth=2, 
       label=f'Fit: ΔG_O = {coeffs[0]:.2f}ΔG_OH + {coeffs[1]:.2f}')
ax.scatter(x, y, s=150, alpha=0.8, edgecolors='black', c=range(len(x)), cmap='viridis')

# 添加标签
for i, row in df_ads.iterrows():
    if abs(row['dG_OH*']) < 1.0:
        ax.annotate(row['Surface'], (row['dG_OH*'], row['dG_O*']),
                   xytext=(5, 5), textcoords='offset points', fontsize=8)

ax.set_xlabel('ΔG$_{OH^*}$ (eV)', fontsize=14, fontweight='bold')
ax.set_ylabel('ΔG$_{O^*}$ (eV)', fontsize=14, fontweight='bold')
ax.set_title('Scaling Relation: O* vs OH*', fontsize=16, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)

# OOH* vs OH*
ax = axes[1]
x = df_ads['dG_OH*']
y = df_ads['dG_OOH*']

coeffs = np.polyfit(x, y, 1)
ax.plot(x_fit, fit_line(x_fit), 'r--', linewidth=2,
       label=f'Fit: slope={coeffs[0]:.2f}')
ax.plot(x_fit, x_fit + 3.2, 'k:', linewidth=2,
       label='Theory: ΔG$_{OOH}$ = ΔG$_{OH}$ + 3.2')
ax.scatter(x, y, s=150, alpha=0.8, edgecolors='black', c=range(len(x)), cmap='plasma')

ax.set_xlabel('ΔG$_{OH^*}$ (eV)', fontsize=14, fontweight='bold')
ax.set_ylabel('ΔG$_{OOH^*}$ (eV)', fontsize=14, fontweight='bold')
ax.set_title('Scaling Relation: OOH* vs OH*', fontsize=16, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n✓ O* vs OH* 斜率: {coeffs[0]:.3f}")
print(f"✓ OOH* vs OH* 斜率: {coeffs[0]:.3f}")

## 7. Phase 3: 计算过电位

In [ ]:
# 计算过电位
overpotentials = designer.calculate_overpotentials()

# 创建结果数据表
result_data = []
for surface, data in overpotentials.items():
    metal, facet = surface.rsplit('_', 1)
    result_data.append({
        'Surface': surface,
        'Metal': metal,
        'Facet': facet,
        'dG_O*': data['dG_O'],
        'dG_OH*': data['dG_OH'],
        'dG_OOH*': data['dG_OOH'],
        'ORR_η (V)': data['orr']['overpotential'],
        'OER_η (V)': data['oer']['overpotential'],
        'Total_η (V)': data['orr']['overpotential'] + data['oer']['overpotential']
    })

df_results = pd.DataFrame(result_data)
df_results = df_results.sort_values('Total_η (V)')

print("\n过电位计算结果:\n")
display(df_results[['Surface', 'ORR_η (V)', 'OER_η (V)', 'Total_η (V)']].head(15))

## 8. ORR火山图

In [ ]:
# 创建ORR火山图
fig, ax = plt.subplots(figsize=(12, 9))

x = df_results['dG_OH*']
y = df_results['ORR_η (V)']

# 理论火山曲线
x_theory = np.linspace(-0.5, 1.5, 100)
y_theory = np.maximum(0.8 - x_theory, x_theory + 0.2)
y_theory = np.maximum(y_theory, 0.4)

ax.plot(x_theory, y_theory, 'k-', linewidth=2.5, alpha=0.6, label='Theoretical Volcano')

# 按金属着色
unique_metals = df_results['Metal'].unique()
colors = plt.cm.tab20(np.linspace(0, 1, len(unique_metals)))
metal_colors = dict(zip(unique_metals, colors))

for metal in unique_metals:
    mask = df_results['Metal'] == metal
    ax.scatter(x[mask], y[mask], s=200, label=metal, 
              color=metal_colors[metal], alpha=0.8, edgecolors='black', linewidth=1.5)

# 标记最佳催化剂
best_idx = y.idxmin()
best_orr = df_results.loc[best_idx]
ax.scatter(x[best_idx], y[best_idx], s=500, marker='*', 
          color='gold', edgecolors='black', linewidth=2.5,
          label=f"Best: {best_orr['Surface']}", zorder=10)

# 标记已知催化剂
known_catalysts = ['Pt_111', 'Pd_111', 'Au_111']
for cat in known_catalysts:
    if cat in df_results['Surface'].values:
        row = df_results[df_results['Surface'] == cat].iloc[0]
        ax.annotate(cat, (row['dG_OH*'], row['ORR_η (V)']),
                   xytext=(10, 10), textcoords='offset points',
                   fontsize=11, fontweight='bold',
                   bbox=dict(boxstyle='round,pad=0.4', facecolor='yellow', alpha=0.8))

ax.set_xlabel('ΔG$_{OH^*}$ (eV)', fontsize=18, fontweight='bold')
ax.set_ylabel('ORR Overpotential η (V)', fontsize=18, fontweight='bold')
ax.set_title('ORR Activity Volcano Plot\n(DFT-Calculated Adsorption Energies)',
            fontsize=20, fontweight='bold', pad=20)
ax.legend(fontsize=11, ncol=2, loc='upper right')
ax.grid(True, alpha=0.3)
ax.tick_params(labelsize=14)
ax.set_ylim([0, max(y)*1.2])

plt.tight_layout()
plt.savefig(f"{config.work_dir}/ORR_volcano_notebook.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✓ 最佳ORR催化剂: {best_orr['Surface']}")
print(f"   过电位: {best_orr['ORR_η (V)']:.3f} V")

## 9. OER火山图

In [ ]:
# 创建OER火山图
fig, ax = plt.subplots(figsize=(12, 9))

x = df_results['dG_O*'] - df_results['dG_OH*']
y = df_results['OER_η (V)']

# 理论曲线
x_theory = np.linspace(0.5, 2.5, 100)
y_theory = np.maximum(x_theory - 1.6, 2.46 - x_theory)
y_theory = np.maximum(y_theory, 0.3)

ax.plot(x_theory, y_theory, 'k-', linewidth=2.5, alpha=0.6, label='Theoretical Volcano')

# 按晶面着色
facet_markers = {'111': 'o', '100': 's', '110': '^'}
facet_colors = {'111': '#3498db', '100': '#e74c3c', '110': '#2ecc71'}

for facet in df_results['Facet'].unique():
    mask = df_results['Facet'] == facet
    marker = facet_markers.get(facet, 'o')
    color = facet_colors.get(facet, 'gray')
    ax.scatter(x[mask], y[mask], s=200, label=facet, 
              marker=marker, color=color, alpha=0.8, edgecolors='black', linewidth=1.5)

# 标记最佳催化剂
best_idx = y.idxmin()
best_oer = df_results.loc[best_idx]
ax.scatter(x[best_idx], y[best_idx], s=500, marker='*',
          color='gold', edgecolors='black', linewidth=2.5,
          label=f"Best: {best_oer['Surface']}", zorder=10)

ax.set_xlabel('ΔG$_{O^*}$ - ΔG$_{OH^*}$ (eV)', fontsize=18, fontweight='bold')
ax.set_ylabel('OER Overpotential η (V)', fontsize=18, fontweight='bold')
ax.set_title('OER Activity Volcano Plot\n(DFT-Calculated Adsorption Energies)',
            fontsize=20, fontweight='bold', pad=20)
ax.legend(fontsize=13, loc='upper right')
ax.grid(True, alpha=0.3)
ax.tick_params(labelsize=14)
ax.set_ylim([0, max(y)*1.2])

plt.tight_layout()
plt.savefig(f"{config.work_dir}/OER_volcano_notebook.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✓ 最佳OER催化剂: {best_oer['Surface']}")
print(f"   过电位: {best_oer['OER_η (V)']:.3f} V")

## 10. 双功能催化剂活性图

In [ ]:
# 创建双功能活性图
fig, ax = plt.subplots(figsize=(12, 10))

x = df_results['ORR_η (V)']
y = df_results['OER_η (V)']

# 创建背景热力图
x_range = np.linspace(0, max(x)*1.1, 50)
y_range = np.linspace(0, max(y)*1.1, 50)
X, Y = np.meshgrid(x_range, y_range)
Z = 1 / (X + Y + 0.01)

im = ax.contourf(X, Y, Z, levels=20, cmap='RdYlGn', alpha=0.3)

# 绘制催化剂点
for _, row in df_results.iterrows():
    color = '#2ecc71' if row['Total_η (V)'] < 0.8 else \n           '#3498db' if row['Total_η (V)'] < 1.0 else \n           '#f39c12' if row['Total_η (V)'] < 1.5 else '#e74c3c'
    
    size = max(100, 500 / (row['Total_η (V)'] + 0.1))
    ax.scatter(row['ORR_η (V)'], row['OER_η (V)'], 
              s=size, c=color, alpha=0.8, edgecolors='black', linewidth=1.5)

# 标记最佳双功能催化剂
best_bifunc = df_results.loc[df_results['Total_η (V)'].idxmin()]
ax.scatter(best_bifunc['ORR_η (V)'], best_bifunc['OER_η (V)'],
          s=800, marker='*', color='gold', edgecolors='black', linewidth=3,
          label=f"Best: {best_bifunc['Surface']}", zorder=10)

# 添加标签
for _, row in df_results.head(5).iterrows():
    ax.annotate(row['Surface'], (row['ORR_η (V)'], row['OER_η (V)']),
               xytext=(10, 10), textcoords='offset points',
               fontsize=10, fontweight='bold')

# 理想区域
ax.axvline(x=0.4, color='blue', linestyle='--', linewidth=2, alpha=0.7)
ax.axhline(y=0.4, color='red', linestyle='--', linewidth=2, alpha=0.7)
ax.fill_between([0, 0.4], [0, 0], [0.4, 0.4], alpha=0.15, color='green',
               label='Target Region')

ax.set_xlabel('ORR Overpotential η (V)', fontsize=18, fontweight='bold')
ax.set_ylabel('OER Overpotential η (V)', fontsize=18, fontweight='bold')
ax.set_title('Bifunctional Electrocatalyst Activity Map\n(ORR + OER Performance)',
            fontsize=20, fontweight='bold', pad=20)
ax.legend(fontsize=13, loc='upper right')
ax.grid(True, alpha=0.3)
ax.tick_params(labelsize=14)

cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Activity Index', fontsize=14)

plt.tight_layout()
plt.savefig(f"{config.work_dir}/bifunctional_map_notebook.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✓ 最佳双功能催化剂: {best_bifunc['Surface']}")
print(f"   ORR过电位: {best_bifunc['ORR_η (V)']:.3f} V")
print(f"   OER过电位: {best_bifunc['OER_η (V)']:.3f} V")
print(f"   总过电位: {best_bifunc['Total_η (V)']:.3f} V")

## 11. 综合分析

In [ ]:
# 运行完整分析
df_analysis = designer.analyze_results()

print("\n最佳催化剂排名:\n")
display(df_analysis[['Surface', 'ORR_η (V)', 'OER_η (V)', 'Total_η (V)']].head(10))

## 12. 实验验证建议

In [ ]:
# 生成详细报告
designer.generate_report(df_analysis)

print("\n建议优先实验验证的催化剂:\n")
for i, (_, row) in enumerate(df_analysis.head(5).iterrows(), 1):
    print(f"{i}. {row['Surface']}")
    print(f"   - ORR过电位: {row['ORR_η (V)']:.3f} V")
    print(f"   - OER过电位: {row['OER_η (V)']:.3f} V")
    print(f"   - 总过电位: {row['Total_η (V)']:.3f} V")
    if row['Total_η (V)'] < 0.8:
        print(f"   - 评价: 优秀双功能催化剂 ★★★★★")
    elif row['Total_η (V)'] < 1.0:
        print(f"   - 评价: 良好双功能催化剂 ★★★★☆")
    elif row['Total_η (V)'] < 1.5:
        print(f"   - 评价: 一般催化剂 ★★★☆☆")
    print()

## 13. 总结

In [ ]:
print("="*70)
print("电催化剂设计案例研究 - 完成摘要")
print("="*70)
print(f"\n总表面模型: {len(surfaces)}")
print(f"分析金属: {len(df_results['Metal'].unique())}")
print(f"分析晶面: {len(df_results['Facet'].unique())}")

best_orr = df_analysis.loc[df_analysis['ORR_η (V)'].idxmin()]
best_oer = df_analysis.loc[df_analysis['OER_η (V)'].idxmin()]
best_bifunc = df_analysis.loc[df_analysis['Total_η (V)'].idxmin()]

print(f"\n最佳ORR催化剂: {best_orr['Surface']}")
print(f"  - 过电位: {best_orr['ORR_η (V)']:.3f} V")
print(f"  - dG_OH*: {best_orr['dG_OH*']:.3f} eV")

print(f"\n最佳OER催化剂: {best_oer['Surface']}")
print(f"  - 过电位: {best_oer['OER_η (V)']:.3f} V")
print(f"  - dG_O* - dG_OH*: {best_oer['dG_O*'] - best_oer['dG_OH*']:.3f} eV")

print(f"\n最佳双功能催化剂: {best_bifunc['Surface']}")
print(f"  - 总过电位: {best_bifunc['Total_η (V)']:.3f} V")
print(f"  - ORR过电位: {best_bifunc['ORR_η (V)']:.3f} V")
print(f"  - OER过电位: {best_bifunc['OER_η (V)']:.3f} V")

print(f"\n所有结果保存在: {Path(config.work_dir).absolute()}")
print("="*70)